# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
fraud = pd.read_csv("card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


# Data Exploration

In [3]:
fraud.shape

(1000000, 8)

In [4]:
fraud.isna().sum()

distance_from_home                0
distance_from_last_transaction    0
ratio_to_median_purchase_price    0
repeat_retailer                   0
used_chip                         0
used_pin_number                   0
online_order                      0
fraud                             0
dtype: int64

In [5]:
fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [6]:
cols = ["repeat_retailer", "used_chip", "used_pin_number", "online_order", "fraud"]
fraud[cols]= fraud[cols].astype("bool")

In [7]:
fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  bool   
 4   used_chip                       1000000 non-null  bool   
 5   used_pin_number                 1000000 non-null  bool   
 6   online_order                    1000000 non-null  bool   
 7   fraud                           1000000 non-null  bool   
dtypes: bool(5), float64(3)
memory usage: 27.7 MB


In [8]:
fraud.describe()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price
count,1000000.000000,1000000.000000,1000000.000000
mean,26.628792,5.036519,1.824182
std,65.390784,25.843093,2.799589
min,0.004874,0.000118,0.004399
25%,3.878008,0.296671,0.475673
50%,9.967760,0.998650,0.997717
75%,25.743985,3.355748,2.096370
max,10632.723672,11851.104565,267.802942


In [9]:
fraud.value_counts()

distance_from_home  distance_from_last_transaction  ratio_to_median_purchase_price  repeat_retailer  used_chip  used_pin_number  online_order  fraud
0.004874            0.198102                        0.998148                        False            False      False            True          False    1
18.271839           0.558001                        0.161158                        True             False      False            True          False    1
18.271944           0.079844                        0.264669                        True             False      False            False         False    1
18.271993           1.078692                        0.712373                        True             False      False            True          False    1
18.272016           0.197228                        0.615863                        True             False      False            False         False    1
                                                                                 

**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

## Distribution of target variable

In [10]:
fraud["fraud"].value_counts(normalize=True)

fraud
False    0.912597
True     0.087403
Name: proportion, dtype: float64

- Dataset is heavily imbalanced with a ratio of 92-8 between negative to positive class

# Train Test split

In [11]:
X = fraud.drop("fraud", axis=1)
y = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1, #for large datasets, we can use a smaller test size
    random_state=42, #for reproducibility
    stratify=y #for maintaining imbalance in train and test sets
    )

# 1. Logistic Regression (basic)

In [12]:
# Initialize model
lr= LogisticRegression()

# Train
lr.fit(X_train, y_train)

# Predict
y_pred = lr.predict(X_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Evaluation

In [13]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# predicted probablities
# Probability of positive class (Revenue = True)
y_train_proba = lr.predict_proba(X_train)[:, 1]
y_test_proba  = lr.predict_proba(X_test)[:, 1]

roc_auc_train = roc_auc_score(y_train, y_train_proba)
roc_auc_test  = roc_auc_score(y_test, y_test_proba)

print("ROC AUC - Train:", roc_auc_train)
print("ROC AUC - Test :", roc_auc_test)

pr_auc_train = average_precision_score(y_train, y_train_proba)
pr_auc_test  = average_precision_score(y_test, y_test_proba)

print("PR AUC  - Train:", pr_auc_train)
print("PR AUC  - Test :", pr_auc_test)

[[90655   605]
 [ 3628  5112]]
              precision    recall  f1-score   support

       False       0.96      0.99      0.98     91260
        True       0.89      0.58      0.71      8740

    accuracy                           0.96    100000
   macro avg       0.93      0.79      0.84    100000
weighted avg       0.96      0.96      0.95    100000

ROC AUC - Train: 0.9656470500551089
ROC AUC - Test : 0.9673434903970902
PR AUC  - Train: 0.7986373770082712
PR AUC  - Test : 0.8069777272075808


- The basic model catches 58% of fraudulent transactions at a precision of 89%
- The model is not overfitted, as we see that the AUC's are quite close to each others' values for train and test sets

# 2. Oversampling (Random)

In [14]:
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train) 

lr_over = LogisticRegression(max_iter=5000, random_state=42)

# Train
lr_over.fit(X_train_over, y_train_over)

# Predict
y_pred = lr_over.predict(X_test)

## Evaluation

In [16]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# predicted probablities
# Probability of positive class (Revenue = True)
y_train_over_proba = lr_over.predict_proba(X_train_over)[:, 1]
y_test_over_proba  = lr_over.predict_proba(X_test)[:, 1]

roc_auc_train = roc_auc_score(y_train_over, y_train_over_proba)
roc_auc_test  = roc_auc_score(y_test, y_test_over_proba)

print("ROC AUC - Train:", roc_auc_train)
print("ROC AUC - Test :", roc_auc_test)

pr_auc_train = average_precision_score(y_train_over, y_train_over_proba)
pr_auc_test  = average_precision_score(y_test, y_test_over_proba)

print("PR AUC  - Train:", pr_auc_train)
print("PR AUC  - Test :", pr_auc_test)

[[85141  6119]
 [  442  8298]]
              precision    recall  f1-score   support

       False       0.99      0.93      0.96     91260
        True       0.58      0.95      0.72      8740

    accuracy                           0.93    100000
   macro avg       0.79      0.94      0.84    100000
weighted avg       0.96      0.93      0.94    100000

ROC AUC - Train: 0.9793319988433289
ROC AUC - Test : 0.9799973821871375
PR AUC  - Train: 0.9673994890861715
PR AUC  - Test : 0.7604085077136198


- The model with oversampling catches 95% of fraudulent transactions at a precision of 58%

# 3. Undersampling (Random)

In [21]:
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

lr_under = LogisticRegression(max_iter=5000, random_state=42)
lr_under.fit(X_train_under, y_train_under)
y_pred = lr_under.predict(X_test)

## Evaluation

In [22]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# predicted probablities
# Probability of positive class (Revenue = True)
y_train_under_proba = lr_under.predict_proba(X_train_under)[:, 1]
y_test_under_proba  = lr_under.predict_proba(X_test)[:, 1]

roc_auc_train = roc_auc_score(y_train_under, y_train_under_proba)
roc_auc_test  = roc_auc_score(y_test, y_test_under_proba)

print("ROC AUC - Train:", roc_auc_train)
print("ROC AUC - Test :", roc_auc_test)

pr_auc_train = average_precision_score(y_train_under, y_train_under_proba)
pr_auc_test  = average_precision_score(y_test, y_test_under_proba)

print("PR AUC  - Train:", pr_auc_train)
print("PR AUC  - Test :", pr_auc_test)

[[85126  6134]
 [  442  8298]]
              precision    recall  f1-score   support

       False       0.99      0.93      0.96     91260
        True       0.57      0.95      0.72      8740

    accuracy                           0.93    100000
   macro avg       0.78      0.94      0.84    100000
weighted avg       0.96      0.93      0.94    100000

ROC AUC - Train: 0.9792084042579485
ROC AUC - Test : 0.980005138460736
PR AUC  - Train: 0.9669978783625088
PR AUC  - Test : 0.7603753341087095


- The model with undersampling catches 95% of fraudulent transactions at a precision of 57%

# 4. SMOTE

In [23]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

lr_smote = LogisticRegression(max_iter=5000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred = lr_smote.predict(X_test)

## Evaluation

In [25]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# predicted probablities
# Probability of positive class (Revenue = True)
y_train_smote_proba = lr_smote.predict_proba(X_train_smote)[:, 1]
y_test_smote_proba  = lr_smote.predict_proba(X_test)[:, 1]

roc_auc_train = roc_auc_score(y_train_smote, y_train_smote_proba)
roc_auc_test  = roc_auc_score(y_test, y_test_smote_proba)

print("ROC AUC - Train:", roc_auc_train)
print("ROC AUC - Test :", roc_auc_test)

pr_auc_train = average_precision_score(y_train_smote, y_train_smote_proba)
pr_auc_test  = average_precision_score(y_test, y_test_smote_proba)

print("PR AUC  - Train:", pr_auc_train)
print("PR AUC  - Test :", pr_auc_test)

[[85136  6124]
 [  448  8292]]
              precision    recall  f1-score   support

       False       0.99      0.93      0.96     91260
        True       0.58      0.95      0.72      8740

    accuracy                           0.93    100000
   macro avg       0.78      0.94      0.84    100000
weighted avg       0.96      0.93      0.94    100000

ROC AUC - Train: 0.9792758835150541
ROC AUC - Test : 0.9799291178271552
PR AUC  - Train: 0.9674568325055994
PR AUC  - Test : 0.7618249084251896


- The model with undersampling catches 95% of fraudulent transactions at a precision of 58%

# Conclusion

- The baseline logistic regression model achieves high precision but low recall, missing many fraud cases
- Among the balancing methods, SMOTE has the best trade-off because it improves recall (95%) while maintaining a high ROC-AUC of 98%